# PyTorch Dataset

Time to refresh your PyTorch Datasets knowledge!

Before model training can commence, you need to load the data and pass it to the model in the right format. In PyTorch, this is handled by Datasets and DataLoaders. Let's start with building a PyTorch Dataset for our water potability data.

In [1]:
import pandas as pd
from torch.utils.data import Dataset

In [3]:
class WaterDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        # Load data to pandas DataFrame
        df = pd.read_csv(csv_path)
        # Convert data to a NumPy array and assign to self.data
        self.data = df.to_numpy()
        
    # Implement __len__ to return the number of data samples
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        features = self.data[idx, :-1]
        # Assign last data column to label
        label = self.data[idx, -1]
        return features, label

# PyTorch DataLoader

Good job defining the Dataset class! The WaterDataset you just created is now available for you to use.

The next step in preparing the training data is to set up a DataLoader. A PyTorch DataLoader can be created from a Dataset to load data, split it into batches, and perform transformations on the data if desired. Then, it yields a data sample ready for training.

In [5]:
from torch.utils.data import Dataset, DataLoader

# Create an instance of the WaterDataset
dataset_train = WaterDataset("dataset/electricity_train.csv")

# Create a DataLoader based on dataset_train
dataloader_train = DataLoader(
    dataset_train,
    batch_size=2,
    shuffle=True,
)

# Get a batch of features and labels
features, labels = next(iter(dataloader_train))
print(features, labels)

['2011-01-01 00:15:00'] -0.7043185184993116


# PyTorch Model

You will use the OOP approach to define the model architecture. Recall that this requires setting up a model class and defining two methods inside it:

.__init__(), in which you define the layers you want to use;

forward(), in which you define what happens to the model inputs once it receives them; this is where you pass inputs through pre-defined layers.

Let's build a model with three linear layers and ReLU activations. After the last linear layer, you need a sigmoid activation instead, which is well-suited for binary classification tasks like our water potability prediction problem.

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define the three linear layers
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        
    def forward(self, x):
        # Pass x through linear layers adding activations
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

# Training loop

Time to refresh your knowledge on training loops! Let's train a classifier to predict water potability. You will use the model called net, which you built in the previous lesson.

The training loop code has already been written but, unfortunately, all the code lines got mixed up! Can you sort them to recover the proper order of operations performed during training?

<center><img src="images/01.04.png"  style="width: 400px, height: 300px;"/></center>


# Optimizers

It's time to explore the different optimizers that you can use for training your model.

In [8]:
def train_model(optimizer, net, num_epochs):
    criterion = nn.BCELoss()
    for epoch in range(num_epochs):
        running_loss = 0.
        for features, labels in dataloader_train:
            optimizer.zero_grad()
            outputs = net(features)
            loss = criterion(outputs, labels.view(-1, 1))
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    train_loss = running_loss / len(dataloader_train)
    print(f"Training loss after {num_epochs} epochs: {train_loss}")

In [10]:
# import torch.optim as optim

# net = Net()

# # Define the SGD optimizer
# optimizer = optim.SGD(net.parameters(), lr=0.001)

# train_model(
#     optimizer=optimizer,
#     net=net,
#     num_epochs=10,
# )

In [11]:
# import torch.optim as optim

# net = Net()

# # Define the RMSprop optimizer
# optimizer = optim.RMSprop(net.parameters(), lr=0.001)

# train_model(
#     optimizer=optimizer,
#     net=net,
#     num_epochs=10,
# )

In [13]:
# import torch.optim as optim

net = Net()

# # Define the Adam optimizer
# optimizer = optim.Adam(net.parameters(), lr=0.001)

# train_model(
#     optimizer=optimizer,
#     net=net,
#     num_epochs=10,
# )

# Model evaluation

With the training loop sorted out, you have trained the model for 1000 epochs, and it is available to you as net. You have also set up a test_dataloader in exactly the same way as you did with train_dataloader before—just reading the data from the test rather than the train directory.

You can now evaluate the model on test data. To do this, you will need to write the evaluation loop to iterate over the batches of test data, get the model's predictions for each batch, and calculate the accuracy score for it. Let's do it!

In [15]:
# import torch
# from torchmetrics import Accuracy

# # Set up binary accuracy metric
# acc = Accuracy(task="binary")

# net.eval()
# with torch.no_grad():
#     for features, labels in dataloader_test:
#         # Get predicted probabilities for test data batch
#         outputs = net(features)
#         preds = (outputs >= 0.5).float()
#         acc(preds, labels.view(-1, 1))

# # Compute total test accuracy
# test_accuracy = acc.compute()
# print(f"Test accuracy: {test_accuracy}")

# Initialization and activation

The problems of unstable (vanishing or exploding) gradients are a challenge that often arises in training deep neural networks. In this and the following exercises, you will expand the model architecture that you built for the water potability classification task to make it more immune to those problems.

As a first step, you'll improve the weights initialization by using He (Kaiming) initialization strategy. To do so, you will need to call the proper initializer from the torch.nn.init module, which has been imported for you as init. Next, you will update the activations functions from the default ReLU to the often better ELU.

In [17]:
from torch.nn import init
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        
        # Apply He initialization
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight, nonlinearity="sigmoid")

    def forward(self, x):
        # Update ReLU activation to ELU
        x = nn.functional.elu(self.fc1(x))
        x = nn.functional.elu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

# Activations: ReLU vs. ELU

The choice of the activation functions used in the model (combined with the corresponding weight initialization) can have a strong impact on the training process. In particular, the proper activation can prevent the network from experiencing unstable gradients problems.

In the previous exercise, you have switched from ReLU to ELU activations. Do you remember which characteristics of the two activations justify this change?

<center><img src="images/01.05.png"  style="width: 400px, height: 300px;"/></center>


# Batch Normalization

As a final improvement to the model architecture, let's add the batch normalization layer after each of the two linear layers. The batch norm trick tends to accelerate training convergence and protects the model from vanishing and exploding gradients issues.

In [19]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        # Add two batch normalization layers
        self.bn1 = nn.BatchNorm1d(16)
        self.bn2 = nn.BatchNorm1d(8)
        
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight, nonlinearity="sigmoid") 
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = nn.functional.elu(x)

        # Pass x through the second set of layers
        x = self.fc2(x)
        x = self.bn2(x)
        x = nn.functional.elu(x)

        x = nn.functional.sigmoid(self.fc3(x))
        return x